# Day 6 - Model Evaluation and Threshold Tuning

Goals:
- Evaluate LightGBM performance
- Tune probability thresholds
- Analyze precision vs recall tradeoffs
- Generate business-focused evaluation metrics
- Save the trained model for deployment

In [4]:
# day 6 - model evaluation notebook
# i’m loading the final modeling table i already built so i don’t have to redo feature engineering

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    f1_score
)

# load the saved modeling dataset
model_df = pd.read_parquet("../data/processed/modeling_dataset.parquet")

print("modeling dataframe loaded")
print("shape:", model_df.shape)
print()
print(model_df.head())

modeling dataframe loaded
shape: (97811, 30)

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39 2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1       

In [5]:
# split the saved dataset by time
# train = orders before June 1, 2018
# test = orders from June 1, 2018 onward

cutoff_date = pd.Timestamp("2018-06-01")

train_df = model_df[model_df["order_purchase_timestamp"] < cutoff_date].copy()
test_df = model_df[model_df["order_purchase_timestamp"] >= cutoff_date].copy()

feature_cols = [
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "is_weekend",
    "is_month_end",
    "promised_delivery_days",
    "seller_late_rate",
    "total_order_value",
    "total_freight_value",
    "avg_product_weight_g",
    "total_package_volume_cm3",
    "avg_freight_price_ratio",
    "n_items",
]

target_col = "is_late"

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("train rows:", len(train_df))
print("test rows:", len(test_df))
print()
print("train labels:")
print(y_train.value_counts())
print()
print("test labels:")
print(y_test.value_counts())

train rows: 78874
test rows: 18937

train labels:
is_late
0    72999
1     5875
Name: count, dtype: int64

test labels:
is_late
0    18265
1      672
Name: count, dtype: int64


In [6]:
# training the main LightGBM model
# this is a stronger tree-based model compared to logistic regression

from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

# train model
lgbm.fit(X_train, y_train)

# predictions
y_pred = lgbm.predict(X_test)

# probability predictions for ROC-AUC
y_prob = lgbm.predict_proba(X_test)[:, 1]

print("LightGBM trained successfully")
print()

print("ROC-AUC score:")
print(round(roc_auc_score(y_test, y_prob), 4))
print()

print("classification report:")
print(classification_report(y_test, y_pred))
print()

print("confusion matrix:")
print(confusion_matrix(y_test, y_pred))

[LightGBM] [Info] Number of positive: 5875, number of negative: 72999
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003043 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1669
[LightGBM] [Info] Number of data points in the train set: 78874, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [7]:
# testing different probability thresholds
# default threshold is 0.50
# lowering threshold usually increases recall

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.5, 0.4, 0.3, 0.25, 0.2, 0.15]

print("threshold tuning results")
print()

for t in thresholds:

    # convert probabilities into predictions
    y_pred_thresh = (y_prob >= t).astype(int)

    precision = precision_score(y_test, y_pred_thresh)
    recall = recall_score(y_test, y_pred_thresh)
    f1 = f1_score(y_test, y_pred_thresh)

    print(f"threshold = {t}")
    print(f"precision = {precision:.3f}")
    print(f"recall    = {recall:.3f}")
    print(f"f1-score  = {f1:.3f}")
    print("-" * 30)

threshold tuning results

threshold = 0.5
precision = 0.086
recall    = 0.417
f1-score  = 0.143
------------------------------
threshold = 0.4
precision = 0.071
recall    = 0.610
f1-score  = 0.128
------------------------------
threshold = 0.3
precision = 0.061
recall    = 0.789
f1-score  = 0.113
------------------------------
threshold = 0.25
precision = 0.055
recall    = 0.847
f1-score  = 0.104
------------------------------
threshold = 0.2
precision = 0.052
recall    = 0.900
f1-score  = 0.098
------------------------------
threshold = 0.15
precision = 0.049
recall    = 0.957
f1-score  = 0.094
------------------------------


## Threshold Tuning Insights

The default probability threshold of 0.50 produced moderate recall for delayed deliveries.

Lowering the threshold significantly improved recall:

- threshold 0.50 → recall 41.7%
- threshold 0.30 → recall 78.9%
- threshold 0.20 → recall 90.0%

However, higher recall came at the cost of lower precision, meaning more false positive delay alerts.

This demonstrates the classic precision-recall tradeoff in imbalanced classification problems.

In a real logistics business setting, threshold selection would depend on operational priorities:
- prioritize recall if avoiding missed delays is critical
- prioritize precision if intervention costs are high